In [1]:
!pip install datasets tokenizers tiktoken

import os
import time
from datasets import load_dataset
from tokenizers import Tokenizer
import tiktoken

In [2]:
base_dir = "../artifacts"
model_dir = "tokenizer_superbpe_50k_olmo_p99"
tokenizer_path = os.path.join(base_dir, model_dir, "tokenizer.json")

In [11]:
superbpe_tokenizer = Tokenizer.from_file(tokenizer_path) #50k
gpt_tokenizer = tiktoken.get_encoding("r50k_base") #50k
#fair compare with 50k vocab size, so we use r50k_base for gpt tokenizer
gpt_tokenizer100k = tiktoken.get_encoding("cl100k_base") #100k

In [5]:
num_samples = 150000
dataset = load_dataset(
    "HuggingFaceFW/fineweb-edu", 
    name="sample-10BT", 
    split="train", 
    streaming=True
)
subsample = dataset.take(num_samples)

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

In [13]:
texts = [example['text'] for example in subsample]
total_chars = sum(len(text) for text in texts)
total_words = sum(len(text.split()) for text in texts)
print(f"\nTotal characters to process: {total_chars:,}")


Total characters to process: 715,708,437


In [14]:
def evaluate_tokenizer(name, tokenize_fn, texts):
    print(f"\nEvaluating {name}...")
    start_time = time.time()
    
    total_tokens = 0
    for text in texts:
        tokens = tokenize_fn(text)
        total_tokens += len(tokens)
        
    end_time = time.time()
    time_taken = end_time - start_time
    
    # Existing metrics
    chars_per_token = total_chars / total_tokens if total_tokens > 0 else 0
    tokens_per_second = total_tokens / time_taken if time_taken > 0 else 0
    
    # New Fertility metric
    fertility = total_tokens / total_words if total_words > 0 else 0
    
    print("-" * 30)
    print(f"Results for {name}:")
    print(f"Total Tokens:      {total_tokens:,}")
    print(f"Chars per Token:   {chars_per_token:.2f} (Higher is better)")
    print(f"Fertility:         {fertility:.2f} tokens/word (Lower is better)")
    print(f"Encoding Speed:    {tokens_per_second:,.0f} tokens/sec")
    print(f"Total Time:        {time_taken:.2f} seconds")
    print("-" * 30)

In [15]:
evaluate_tokenizer(
    name="Custom Tokenizer (SuperBPE 50k)",
    tokenize_fn=lambda t: superbpe_tokenizer.encode(t).ids,
    texts=texts
)


Evaluating Custom Tokenizer (SuperBPE 50k)...
------------------------------
Results for Custom Tokenizer (SuperBPE 50k):
Total Tokens:      124,639,642
Chars per Token:   5.74 (Higher is better)
Fertility:         1.07 tokens/word (Lower is better)
Encoding Speed:    250,693 tokens/sec
Total Time:        497.18 seconds
------------------------------


In [16]:
evaluate_tokenizer(
    name="GPT Tokenizer (50k vocab)",
    tokenize_fn=lambda t: gpt_tokenizer.encode(t, allowed_special={"<|endoftext|>"}),
    texts=texts
)


Evaluating GPT Tokenizer (50k vocab)...
------------------------------
Results for GPT Tokenizer (50k vocab):
Total Tokens:      155,655,979
Chars per Token:   4.60 (Higher is better)
Fertility:         1.34 tokens/word (Lower is better)
Encoding Speed:    562,561 tokens/sec
Total Time:        276.69 seconds
------------------------------


In [17]:
evaluate_tokenizer(
    name="GPT Tokenizer (cl100k_base)",
    tokenize_fn=lambda t: gpt_tokenizer100k.encode(t, allowed_special={"<|endoftext|>"}),
    texts=texts
)


Evaluating GPT Tokenizer (cl100k_base)...
------------------------------
Results for GPT Tokenizer (cl100k_base):
Total Tokens:      151,620,478
Chars per Token:   4.72 (Higher is better)
Fertility:         1.30 tokens/word (Lower is better)
Encoding Speed:    2,369,093 tokens/sec
Total Time:        64.00 seconds
------------------------------
